# Protein ↔ MolecularFunction Relation-Wise Merge


## 0. Configuration

In [16]:
import os
import pandas as pd
import numpy as np

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PROTEIN_MOLECULARFUNCTION/ALL_PROTEIN_MOLECULARFUNCTION.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

In [21]:
# RecName dict (for sources that store UniProt ACs in head)
Uniprot_Recname = pd.read_csv(DB_DIR + 'uniprot/Uniprot_ID_Recname.csv')
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))

# Parsed TrEMBL dict (AC → NAME) for head-name fallback
uniprot = pd.read_csv(DB_DIR + 'uniprot/uniprot_parsed_results.csv')
uniprot_trembl_AC_Name_dict = dict(zip(uniprot['AC'], uniprot['NAME']))
del uniprot

print(f"UniProt RecName dict: {len(Uniprot_Recname_dict):,} | TrEMBL dict: {len(uniprot_trembl_AC_Name_dict):,}")

UniProt RecName dict: 794,151 | TrEMBL dict: 252,635,149


## 1. Load KG Sources

### 1a. CKG

In [5]:
CKG_Protein_MF = pd.read_csv(PROC_DIR + 'CKG/File_18_protein_MolecularFunction_CKG.csv')
CKG_Protein_MF.columns = CKG_Protein_MF.columns.str.lower()
CKG_Protein_MF.rename(columns={'head_alt_name': 'head_detail_name'}, inplace=True)
CKG_Protein_MF['tail_id_is'] = 'Quick_GO'
print(f"CKG:      {len(CKG_Protein_MF):,} rows")
CKG_Protein_MF['kg_source'] = 'CKG'
CKG_Protein_MF['kg_type'] = 'Generalised'
CKG_Protein_MF['species'] = 'Homo sapiens'
CKG_Protein_MF.head(2)

CKG:      220,704 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_go_name,head_id_is,tail_id_is,evidence_type,score,tail_detail_name
0,O94778,ASSOCIATED_WITH,GO:0015265,Protein,Ontology,UniProt,CKG,Aquaporin-8 {ECO:0000303|PubMed:34292591},urea channel activity,Uniprot_ID,Quick_GO,ISS,5,urea channel activity
1,Q9NQ69,ASSOCIATED_WITH,GO:0003677,Protein,Ontology,UniProt,CKG,LIM/homeobox protein Lhx9,DNA binding,Uniprot_ID,Quick_GO,IEA,5,DNA binding


### 1b. CrossBAR

In [6]:
CrossBAR_Protein_MF = pd.read_csv(PROC_DIR + 'crossbar/Protein_MolecularFunction_Crossbar.csv')
CrossBAR_Protein_MF.columns = CrossBAR_Protein_MF.columns.str.lower()
CrossBAR_Protein_MF['tail_id_is'] = 'Quick_GO'
print(f"CrossBAR: {len(CrossBAR_Protein_MF):,} rows")
CrossBAR_Protein_MF['kg_source'] = 'crossbar'
CrossBAR_Protein_MF['kg_type'] = 'Generalised'
CrossBAR_Protein_MF['species'] = 'Homo sapiens'
CrossBAR_Protein_MF.head(2)

CrossBAR: 1,138,818 rows


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,A1DG72,Protein_MolecularFunction,GO:0004222,Protein,MolecularFunction,crossbar,Mitochondrial inner membrane protease atp23,metalloendopeptidase activity,Uniprot_ID,Quick_GO,Generalised,Homo sapiens
1,A1DG72,Protein_MolecularFunction,GO:0008233,Protein,MolecularFunction,crossbar,Mitochondrial inner membrane protease atp23,peptidase activity,Uniprot_ID,Quick_GO,Generalised,Homo sapiens


### 1c. Tarkg

In [7]:
tarkg_protein_MF = pd.read_csv(PROC_DIR + 'TARKG/Protein_MolecularFunction.csv')
tarkg_protein_MF.columns = tarkg_protein_MF.columns.str.lower()
tarkg_protein_MF['tail_id_is'] = 'Quick_GO'
print(f"CrossBAR: {len(tarkg_protein_MF):,} rows")
tarkg_protein_MF['kg_source'] = 'TARKG'
tarkg_protein_MF['kg_type'] = 'Generalised'
tarkg_protein_MF['species'] = 'Homo sapiens'
tarkg_protein_MF.head(2)

CrossBAR: 268,404 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,Q8WYP5,Protein_Molecular Function,GO:0000981,Protein,GO_BP,Molecular Function,TARKG,Protein ELYS,"DNA-binding transcription factor activity, RNA...",Uniprot_ID,Quick_GO,BioKG-2328462,BioKG,0,Generalised,Homo sapiens
1,Q12986,Protein_Molecular Function,GO:0000981,Protein,GO_BP,Molecular Function,TARKG,Transcriptional repressor NF-X1,"DNA-binding transcription factor activity, RNA...",Uniprot_ID,Quick_GO,BioKG-2411286,BioKG,0,Generalised,Homo sapiens


## 2. Check and Fix Duplicate Columns

In [8]:
SOURCE_DFS = [
    ('CKG_Protein_MF',      CKG_Protein_MF),
    ('CrossBAR_Protein_MF', CrossBAR_Protein_MF),
    ('tarkg_protein_MF', tarkg_protein_MF),
    
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[CKG_Protein_MF] ✓ no duplicates
[CrossBAR_Protein_MF] ✓ no duplicates
[tarkg_protein_MF] ✓ no duplicates


## 3. Align Schemas and Concatenate

In [9]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    tmp['head_id_is'] = tmp['head_id_is'].astype(str)
    tmp['tail_id_is'] = tmp['tail_id_is'].astype(str)
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 1,627,926 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,O94778,ASSOCIATED_WITH,GO:0015265,Protein,NaN,Ontology,CKG,Uniprot_ID,Quick_GO,Aquaporin-8 {ECO:0000303|PubMed:34292591},urea channel activity,NaN,NaN
1,Q9NQ69,ASSOCIATED_WITH,GO:0003677,Protein,NaN,Ontology,CKG,Uniprot_ID,Quick_GO,LIM/homeobox protein Lhx9,DNA binding,NaN,NaN


## 4. Standardise Fixed-Value Columns

In [10]:
final_df['head_type'] = 'Protein'
final_df['tail_type'] = 'MolecularFunction'
final_df['relation']  = 'Protein_MolecularFunction'

print("Unique relation:",  set(final_df['relation']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Protein_MolecularFunction'}
Unique tail_id_is: {'Quick_GO'}
Unique kg_source: {'CKG', 'TARKG', 'crossbar'}


## 5. Deduplicate and Add Schema Columns

In [12]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
final_df_dedup.head()

After deduplication: 1,172,768 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A0A011QK89,Protein_MolecularFunction,GO:0016491,Protein,None,MolecularFunction,crossbar,Uniprot_ID,Quick_GO,None,oxidoreductase activity,Generalised,Homo sapiens
1,A0A011QK89,Protein_MolecularFunction,GO:0047545,Protein,None,MolecularFunction,crossbar,Uniprot_ID,Quick_GO,None,2-hydroxyglutarate dehydrogenase activity,Generalised,Homo sapiens
2,A0A024QYT3,Protein_MolecularFunction,GO:0005509,Protein,None,MolecularFunction,crossbar,Uniprot_ID,Quick_GO,None,calcium ion binding,Generalised,Homo sapiens
3,A0A024QYT3,Protein_MolecularFunction,GO:0046872,Protein,None,MolecularFunction,crossbar,Uniprot_ID,Quick_GO,None,metal ion binding,Generalised,Homo sapiens
4,A0A024RBG1,Protein_MolecularFunction,GO:0000298,Protein,None,MolecularFunction,CKG::crossbar,Uniprot_ID,Quick_GO,Diphosphoinositol polyphosphate phosphohydrola...,endopolyphosphatase activity,Generalised,Homo sapiens


In [20]:
cols = [
    'relation', 'head_type', 'relation_type',
    'tail_type', 'kg_source', 'head_id_is',
    'tail_id_is', 'kg_type', 'species'
]

unique_vals = {
    col: set(final_df[col].dropna())
    for col in cols
}

unique_vals

{'relation': {'Protein_MolecularFunction'},
 'head_type': {'Protein'},
 'relation_type': {'GO_BP'},
 'tail_type': {'MolecularFunction'},
 'kg_source': {'CKG', 'TARKG', 'crossbar'},
 'head_id_is': {'Uniprot_ID'},
 'tail_id_is': {'Quick_GO'},
 'kg_type': {'Generalised'},
 'species': {'Homo sapiens'}}

## 6. QC — NaN Check

In [15]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,904364,0,904364
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,869142,0,869142


In [24]:
final_df_dedup['head_detail_name'] = (
    final_df_dedup['head_detail_name']
    .fillna(final_df_dedup['head'].map(uniprot_trembl_AC_Name_dict))
    .fillna(final_df_dedup['head'].map(Uniprot_Recname_dict))
)
                                                                        

NameError: name 'final_df_dedupfinal_df_dedup' is not defined

In [26]:
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A0A011QK89,Protein_MolecularFunction,GO:0016491,Protein,None,MolecularFunction,crossbar,Uniprot_ID,Quick_GO,L-2-hydroxyglutarate dehydrogenase {ECO:0000305},oxidoreductase activity,Generalised,Homo sapiens
1,A0A011QK89,Protein_MolecularFunction,GO:0047545,Protein,None,MolecularFunction,crossbar,Uniprot_ID,Quick_GO,L-2-hydroxyglutarate dehydrogenase {ECO:0000305},2-hydroxyglutarate dehydrogenase activity,Generalised,Homo sapiens
2,A0A024QYT3,Protein_MolecularFunction,GO:0005509,Protein,None,MolecularFunction,crossbar,Uniprot_ID,Quick_GO,Osteocalcin 2a {ECO:0000312|EMBL:DAA64603.1},calcium ion binding,Generalised,Homo sapiens
3,A0A024QYT3,Protein_MolecularFunction,GO:0046872,Protein,None,MolecularFunction,crossbar,Uniprot_ID,Quick_GO,Osteocalcin 2a {ECO:0000312|EMBL:DAA64603.1},metal ion binding,Generalised,Homo sapiens
4,A0A024RBG1,Protein_MolecularFunction,GO:0000298,Protein,None,MolecularFunction,CKG::crossbar,Uniprot_ID,Quick_GO,Diphosphoinositol polyphosphate phosphohydrola...,endopolyphosphatase activity,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1172763,X5M8U1,Protein_MolecularFunction,GO:0016829,Protein,None,MolecularFunction,crossbar,Uniprot_ID,Quick_GO,Receptor-type guanylate cyclase gcy-17 {ECO:00...,lyase activity,Generalised,Homo sapiens
1172764,X5M8U1,Protein_MolecularFunction,GO:0016849,Protein,None,MolecularFunction,crossbar,Uniprot_ID,Quick_GO,Receptor-type guanylate cyclase gcy-17 {ECO:00...,phosphorus-oxygen lyase activity,Generalised,Homo sapiens
1172765,X6R8R1,Protein_MolecularFunction,GO:0000149,Protein,None,MolecularFunction,CKG,Uniprot_ID,Quick_GO,Synaptotagmin-15B {ECO:0000305},SNARE binding,,None
1172766,X6R8R1,Protein_MolecularFunction,GO:0005544,Protein,None,MolecularFunction,CKG,Uniprot_ID,Quick_GO,Synaptotagmin-15B {ECO:0000305},calcium-dependent phospholipid binding,,None


In [27]:
final_df_dedup[final_df_dedup['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


In [25]:
#

## 7. Save Output

In [28]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 1,172,768 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PROTEIN_MOLECULARFUNCTION/ALL_PROTEIN_MOLECULARFUNCTION.csv
